# 03 · Human Review Agent

**Role in the pipeline:** the mandatory checkpoint. Nothing reaches a customer without a human decision.

This notebook:
1. Picks up the Analyst's scores for the latest run
2. Retrieves account context (RAG) and drafts outreach with the **Writer agent** + guardrails
3. Gives you a **review console** inside VS Code: approve / needs edit / reject
4. Logs every decision to the audit table and shows the evaluation metrics

Decisions go through the same FastAPI endpoint the Streamlit frontend uses, so behaviour is identical to production.

In [ ]:
# --- Setup: point the notebook at the project's `copilot` package ---------
# Must run BEFORE any `copilot` import: settings are read once from env vars.
import os, sys, json
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "copilot").exists():          # works from notebooks/ or project root
    if ROOT == ROOT.parent:
        raise RuntimeError("Open the cs_ai_copilot_dss folder in VS Code and run from there.")
    ROOT = ROOT.parent
os.chdir(ROOT)                                            # playbook path is relative to project root
sys.path.insert(0, str(ROOT / "src"))

DEV = ROOT / "notebooks" / "_dev"                         # local, git-ignorable sandbox
DEV.mkdir(parents=True, exist_ok=True)
os.environ["DATABASE_URL"] = f"sqlite:///{(DEV / 'copilot_dev.db').as_posix()}"
os.environ["MODEL_DIR"] = str(DEV / "models")
os.environ.pop("CHROMA_HOST", None)                       # notebooks use the in-memory RAG store

import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 80)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("Project root:", ROOT)
print("Dev database:", os.environ["DATABASE_URL"])

## 1 · Load the latest Analyst run

In [ ]:
from sqlalchemy import select
from copilot.db import Account, Draft, ReviewEvent, RiskScore, SessionLocal, init_db
from copilot.feedback import REASONS, prompt_block, training_records, writer_feedback
from copilot.agents.analyst import AccountAssessment

init_db()
run_file = DEV / "last_run.txt"
if not run_file.exists():
    raise RuntimeError("Run 02_analyst_agent.ipynb first — it creates the scores this notebook reviews.")
RUN_ID = run_file.read_text().strip()

with SessionLocal() as session:
    rows = session.execute(select(RiskScore, Account).join(Account)
                           .where(RiskScore.run_id == RUN_ID)).all()
assessments = [(AccountAssessment(r.account_id, r.clv_score, r.churn_probability,
                                  round(r.churn_probability * r.clv_score, 2), r.risk_level, r.top_factors), acc)
               for r, acc in rows]
assessments.sort(key=lambda t: -t[0].revenue_at_risk)
print(f"Run {RUN_ID}: {len(assessments)} scored accounts")

## 2 · Index customer context (RAG)
Demo documents (emails, call notes, tickets, contract terms). ~15% of accounts deliberately have **no** context, to show the guardrail.

In [ ]:
from copilot.data import load_gold
from copilot.rag import ContextRetriever, synthetic_context

gold = load_gold()
retriever = ContextRetriever()
docs = synthetic_context(gold)
retriever.index(docs)
print(f"Indexed {len(docs)} context documents for {len({d['account_id'] for d in docs})} accounts")
pd.DataFrame(docs).head(6)[["account_id", "doc_type", "text"]]

## 3 · Writer agent → drafts
- With `ANTHROPIC_API_KEY` set in `.env`, drafts are written by Claude and **must cite** retrieved documents.
- Without a key, a deterministic template is used that only restates retrieved context.
- No context → `insufficient_data`; no text is invented.

`MAX_DRAFTS` keeps the demo queue manageable.

In [ ]:
from copilot.agents.nba import recommend_next_best_action
from copilot.agents.writer import WriterAgent
from copilot.config import get_settings

MAX_DRAFTS = 15
s = get_settings()
writer = WriterAgent(os.getenv("ANTHROPIC_API_KEY") or s.anthropic_api_key, s.writer_model, s.min_context_docs)
print("Writer mode:", writer.label)

with SessionLocal() as session:
    already = set(session.scalars(select(Draft.account_id).where(Draft.run_id == RUN_ID)).all())
    feedback_by_action = {}                     # feedback log: approved examples + corrections
    counts = {"pending_review": 0, "insufficient_data": 0, "no_action": 0, "already_drafted": 0}
    for a, acc in assessments[:MAX_DRAFTS]:
        if a.account_id in already:
            counts["already_drafted"] += 1; continue
        rec = recommend_next_best_action(a.risk_level, acc.segment, a.top_factors, s.playbook_path)
        context = retriever.retrieve(a.account_id, f"{rec.action} " + " ".join(f["label"] for f in a.top_factors[:3]))
        if rec.action not in feedback_by_action:
            feedback_by_action[rec.action] = writer_feedback(session, rec.action)
        d = writer.draft(acc.name, a, rec, context, feedback=feedback_by_action[rec.action])
        counts[d.status] += 1
        if d.status == "no_action":
            continue
        session.add(Draft(run_id=RUN_ID, account_id=a.account_id, action=rec.action,
                          action_rationale=rec.rationale, subject=d.subject, body=d.body,
                          context_used=[{"text": c["text"], **c["metadata"]} for c in d.context_used],
                          status=d.status, feedback_used=d.feedback_used))
    session.commit()
    fb_any = writer_feedback(session, "")
print(counts)
print(f"Feedback log: {len(fb_any.examples)} approved examples, {len(fb_any.lessons)} corrections available")

### Guardrail check — accounts held back

In [ ]:
with SessionLocal() as session:
    held = session.execute(select(Draft.account_id, Draft.body)
                           .where(Draft.run_id == RUN_ID, Draft.status == "insufficient_data")).all()
pd.DataFrame(held, columns=["account_id", "reason"]) if held else print("None in this batch.")

## 4 · Review console
The API runs **in-process** here (FastAPI `TestClient`), so decisions use exactly the production endpoint: `POST /drafts/{id}/review`.

In [ ]:
from fastapi.testclient import TestClient
from copilot.api.main import app

api = TestClient(app)
queue = [d for d in api.get("/drafts", params={"status": "pending_review"}).json()]
print(f"{len(queue)} drafts pending review")

### Option A — interactive (ipywidgets)
Step through the queue. Edit the text before approving if you want: the **approved text becomes an example** for future drafts. For *Needs edit* / *Reject* you must choose a **reason** and write an **explanation**: they become corrections for the writer and part of the training dataset.

> If buttons don't render in VS Code: `pip install ipywidgets`, then reload the window. Option B below works without widgets.

In [ ]:
import ipywidgets as w
from IPython.display import display, clear_output

REVIEWER = "Reviewer"
state = {"i": 0}
out = w.Output()
body_box = w.Textarea(layout=w.Layout(width="100%", height="230px"))
reason_box = w.Dropdown(options=[("— reason (required for Needs edit / Reject) —", None)] +
                        [(label, code) for code, label in REASONS.items()], layout=w.Layout(width="100%"))
comment_box = w.Textarea(placeholder="Explanation (required for Needs edit / Reject): what was wrong?",
                         layout=w.Layout(width="100%", height="60px"))
buttons = {k: w.Button(description=k.replace("_", " ").title(), button_style=s, layout=w.Layout(width="140px"))
           for k, s in [("approved", "success"), ("needs_edit", "warning"), ("rejected", "danger")]}

def show():
    with out:
        clear_output()
        if state["i"] >= len(queue):
            print("✔ Queue empty — run Section 5 for metrics."); return
        d = queue[state["i"]]
        risk = api.get(f"/accounts/{d['account_id']}/risk").json()
        expl = api.get(f"/accounts/{d['account_id']}/explanation").json()
        print(f"[{state['i']+1}/{len(queue)}]  {d['account_name']}  ·  {d['account_id']}")
        print(f"Risk {risk['risk_level'].upper()} · churn {risk['churn_probability']:.0%} · CLV €{risk['clv_score']:,.0f}")
        print("Why flagged:", "; ".join(expl["signals"]))
        print("Drivers:", ", ".join(f"{f['label']} ({f['direction']})" for f in expl["top_factors"][:3]))
        print(f"Action: {d['action']} — {d['action_rationale']}")
        print("Grounding:", " | ".join(f"[{c['doc_type']}] {c['text']}" for c in d["context_used"]))
        print("\nSubject:", d["subject"])
        body_box.value = d["body"]; comment_box.value = ""; reason_box.value = None

def decide(decision):
    d = queue[state["i"]]
    payload = {"reviewer": REVIEWER, "decision": decision, "reason": reason_box.value,
               "explanation": comment_box.value or None,
               "edited_body": body_box.value if body_box.value != d["body"] else None}
    r = api.post(f"/drafts/{d['id']}/review", json=payload)
    if not r.ok:                                  # e.g. rejection without a reason: stay on this draft
        with out:
            print("⚠", r.json().get("detail", r.text))
        return
    state["i"] += 1
    show()
    with out:
        print(f"\n→ previous draft: {r.json().get('status', r.text)}")

for k, b in buttons.items():
    b.on_click(lambda _, k=k: decide(k))
display(out, body_box, reason_box, comment_box, w.HBox(list(buttons.values())))
show()

### Option B — scripted review
Useful for batch decisions, testing, or if widgets aren't available. Set `RUN_SCRIPTED = True` to apply.

In [ ]:
RUN_SCRIPTED = False    # True = apply the demo policy below instead of clicking

def review(draft_id: int, decision: str, reason: str | None = None, explanation: str | None = None,
           edited_body: str | None = None, reviewer: str = "Reviewer"):
    r = api.post(f"/drafts/{draft_id}/review", json={"reviewer": reviewer, "decision": decision,
                                                     "reason": reason, "explanation": explanation,
                                                     "edited_body": edited_body})
    return r.status_code, r.json()

if RUN_SCRIPTED:
    pending = api.get("/drafts", params={"status": "pending_review"}).json()
    # Demo policy: approve high-risk drafts, ask for edits on the rest
    for d in pending:
        risk = api.get(f"/accounts/{d['account_id']}/risk").json()["risk_level"]
        if risk == "high":
            print(d["id"], review(d["id"], "approved"))
        else:
            print(d["id"], review(d["id"], "needs_edit", reason="too_long",
                                  explanation="Shorter, and lead with the customer's own concern."))

## 5 · Audit log and evaluation

In [ ]:
with SessionLocal() as session:
    audit = pd.read_sql(select(ReviewEvent.draft_id, ReviewEvent.reviewer, ReviewEvent.decision,
                               ReviewEvent.reason, ReviewEvent.comment.label("explanation"),
                               ReviewEvent.created_at)
                        .order_by(ReviewEvent.created_at.desc()), session.bind)
audit.head(10)

In [ ]:
m = api.get("/metrics").json()
rv = m["review"]
print("Decisions:       ", rv["decisions"])
print("Approval rate:   ", f"{rv['approval_rate']:.0%}" if rv["approval_rate"] is not None else "—")
print("Queue by status: ", m["queue"])

if rv["decisions"]:
    fig, ax = plt.subplots(figsize=(5, 2.6))
    dec = pd.Series(rv["decisions"])
    ax.bar(dec.index, dec.values, color=[{"approved": "#3b8a5a", "needs_edit": "#d0a33b",
                                          "rejected": "#c0503b"}.get(k, "grey") for k in dec.index])
    ax.set_title("Review decisions"); plt.tight_layout(); plt.show()

## 6 · Feedback log: what the writer learns
Approved drafts become **examples** and rejections become **corrections**. Below is exactly what is added to the writer's prompt on the next run (Section 3 picks it up), plus the training dataset.

In [ ]:
import json
with SessionLocal() as session:
    records = training_records(session)
    example_action = queue[0]["action"] if queue else ""
    preview = prompt_block(writer_feedback(session, example_action))
print(f"{len(records)} decisions in the training set "
      f"({sum(r['label'] == 'positive' for r in records)} positive, "
      f"{sum(r['label'] == 'negative' for r in records)} negative)")
print("--- Prompt block for the next draft ---")
print(preview or "Nothing to learn from yet.")

# Training dataset for evaluation / fine-tuning later
out_file = DEV / "writer_feedback.jsonl"
out_file.write_text("\n".join(json.dumps(r, ensure_ascii=False, default=str) for r in records), encoding="utf-8")
print("Saved ->", out_file)

To start a clean review round, delete `notebooks/_dev/` and re-run notebooks 01 → 03.